In [9]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [10]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [11]:
a = pd.read_csv('./number of active users per month according to financial years - Sheet1.csv')
df = a.iloc[1:]
df.columns = ['month','na','ra','ni','ri']
df.set_index('month',inplace = True)
df.head()

,na,ra,ni,ri
month,,,,
19-07,262500.0,0.0,37500.0,0.0
19-08,262500.0,26250.0,37500.0,4500.0
19-09,437500.0,47250.0,62500.0,8250.0
19-10,437500.0,77875.0,62500.0,14250.0
19-11,525000.0,102375.0,75000.0,19000.0


In [15]:
B = pd.read_excel('./Conversion percentage.xlsx')
df1 = B.transpose()
new_header = df1.iloc[1]
df1 = df1[2:]
df1.columns = new_header
df1
df1.columns = ['date','d1','na','ra','d2','ni','ri']
df1.drop(['d1','d2'], axis=1, inplace=True)
df1.set_index('date',inplace = True)
df1

,na,ra,ni,ri
date,,,,
2019-07-19 00:00:00,0.003,0.06,0.006,0.09
2019-08-19 00:00:00,0.005,0.06,0.01,0.09
2019-09-19 00:00:00,0.007,0.06,0.01,0.09
2019-10-19 00:00:00,0.008,0.07,0.012,0.11
2019-11-19 00:00:00,0.008,0.07,0.012,0.11
2019-12-19 00:00:00,0.008,0.07,0.012,0.11
2019-01-20 00:00:00,0.008,0.07,0.012,0.11
2019-02-20 00:00:00,0.008,0.07,0.012,0.11
2019-03-20 00:00:00,0.008,0.07,0.012,0.11


In [13]:
e = df1.loc["Oct-Dec'20":]

split1 = e['na'].tolist()
split2 = e['ra'].tolist()
split3 = e['ni'].tolist()
split4 = e['ri'].tolist()

split1 = [item for item in split1 for i in range(3)]
split2 = [item for item in split2 for i in range(3)]
split3 = [item for item in split3 for i in range(3)]
split4 = [item for item in split4 for i in range(3)]

x11 = pd.DataFrame({'na':split1,'ra':split2,'ni':split3,'ri':split4})
len(x11)

30

In [59]:
x1 = df1.iloc[:15]
y = pd.concat([x1,x11],ignore_index=True)
y.head()

,na,ra,ni,ri
0,0.003,0.06,0.006,0.09
1,0.005,0.06,0.01,0.09
2,0.007,0.06,0.01,0.09
3,0.008,0.07,0.012,0.11
4,0.008,0.07,0.012,0.11


In [63]:
from datetime import timedelta, date
def daterange(date1, date2):
    for n in range(int ((date2 - date1).days)+1):
        yield date1 + timedelta(n)
date_list = []
start_dt = date(19,7,1)
end_dt = date(23,3,30)
for dt in daterange(start_dt, end_dt):
    date_list.append(dt.strftime("%Y-%m"))
    
xy = pd.DataFrame({'months': date_list})
xy.drop_duplicates('months',inplace = True)
xy.reset_index(drop=True, inplace=True)

In [66]:
z = pd.merge(y, xy , left_index=True,right_index = True)
z.set_index('months',inplace=True)
len(z)

45

In [68]:
c3 = pd.DataFrame(df.values*z.values, columns=df.columns, index=df.index)

,na,ra,ni,ri
month,,,,
19-07,787.5,0,225,0
19-08,1312.5,1575,375,405
19-09,3062.5,2835,625,742.5
19-10,3500,5451.25,750,1567.5
19-11,4200,7166.25,900,2090
19-12,4200,8942.5,900,2681.25
20-01,4900,10381.9,1050,3148.75
20-02,4900,12127.5,1050,3685
20-03,4900,13628.1,1050,4097.5


In [73]:
c4 = (c3.sum(axis=1)).round()
c4 = c4.to_frame().reset_index()
c4.set_index('month',inplace=True)
d = pd.merge(c3, c4, left_index=True, right_index=True)
d.columns = ['new android','retained android','new iOS','retained iOS','Total']
d
# c4.columns = ['index','totals']
# c4 = c4[['totals']]
# c4 = c4.set_index(z.index)

,new android,retained android,new iOS,retained iOS,Total
month,,,,,
19-07,787.5,0,225,0,1012.0
19-08,1312.5,1575,375,405,3668.0
19-09,3062.5,2835,625,742.5,7265.0
19-10,3500,5451.25,750,1567.5,11269.0
19-11,4200,7166.25,900,2090,14356.0
19-12,4200,8942.5,900,2681.25,16724.0
20-01,4900,10381.9,1050,3148.75,19481.0
20-02,4900,12127.5,1050,3685,21762.0
20-03,4900,13628.1,1050,4097.5,23676.0
